# Section 1 — Setup and Configuration

In [1]:
import os
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

_HERE = Path.cwd()
for candidate in [_HERE, _HERE.parent, _HERE.parent.parent]:
    if (candidate / 'Phase_1_Machine_Learning').exists() and (candidate / 'Phase_2_Deep_Learning').exists():
        REPO_ROOT = candidate.resolve()
        break
else:
    REPO_ROOT = _HERE.resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Phase_2_Deep_Learning.src.dataset import (
    load_feature_config,
    load_transformers,
    prepare_tensors,
    get_default_paths,
)
from Phase_2_Deep_Learning.src.dl_models import WaterQualityMLP, load_model_checkpoint
from Phase_2_Deep_Learning.src.evaluation import ModelEvaluator
from Phase_2_Deep_Learning.src.training import get_device, compute_regression_metrics

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = get_device()
IG_SAMPLE_SIZE = 1000
IG_STEPS = 50

output_directory = REPO_ROOT / 'Phase_2_Deep_Learning' / 'reports' / 'task_6_outputs'
output_directory.mkdir(parents=True, exist_ok=True)

print(f'Repository Root:  {REPO_ROOT}')
print(f'Active Device:    {device}')
print(f'Output Directory: {output_directory}')

Repository Root:  /Users/amansoni/Documents/Development/Personal Projects/Semester_5_AML_DL
Active Device:    cpu
Output Directory: /Users/amansoni/Documents/Development/Personal Projects/Semester_5_AML_DL/Phase_2_Deep_Learning/reports/task_6_outputs


# Section 2 — Load Data and Models

In [2]:
paths = get_default_paths()
feature_config = load_feature_config(paths['feature_config'])

numerical_features = feature_config['numerical_features']
categorical_features = feature_config['categorical_features']
one_hot_feature_names = feature_config['one_hot_feature_names']
all_feature_names = feature_config['all_feature_names']
target_column = feature_config['target_variable']

test_path = paths['data_dir'] / 'test.parquet'
test_df = pd.read_parquet(test_path)

evaluator = ModelEvaluator(device=device, paths=paths)
evaluator.load_all_models()

print(f'Test Records Ingested: {len(test_df):,} rows × {test_df.shape[1]} columns')
print(f'Numerical Features:    {len(numerical_features)}')
print(f'Categorical Features:  {len(categorical_features)}')
print(f'Encoded OHE Features:  {len(one_hot_feature_names)}')
print(f'Total DL Dimensions:   {len(all_feature_names)}')
print(f'Random Forest Loaded:  {evaluator.rf_model is not None}')
print(f'XGBoost Loaded:        {evaluator.xgb_model is not None}')
print(f'MLP Loaded:            {evaluator.mlp_model is not None}')

✓ RF loaded from /Users/amansoni/Documents/Development/Personal Projects/Semester_5_AML_DL/Phase_1_Machine_Learning/data/GeneratedOutput/03.ModelTraining/random_forest_model.joblib
✓ XGBoost loaded from /Users/amansoni/Documents/Development/Personal Projects/Semester_5_AML_DL/Phase_1_Machine_Learning/data/GeneratedOutput/03.ModelTraining/xgboost_model.json
✓ MLP loaded from /Users/amansoni/Documents/Development/Personal Projects/Semester_5_AML_DL/Phase_2_Deep_Learning/models/mlp_best_checkpoint.pt
Test Records Ingested: 422,943 rows × 45 columns
Numerical Features:    36
Categorical Features:  2
Encoded OHE Features:  17
Total DL Dimensions:   53
Random Forest Loaded:  True
XGBoost Loaded:        True
MLP Loaded:            True


# Section 3 — Compute Predictions on Test Split

In [3]:
t0 = time.time()
test_predictions = evaluator.predict_all(test_df, batch_size=16384)
pred_elapsed = time.time() - t0

y_true = test_predictions['y_true']
y_rf = test_predictions['Random Forest']
y_xgb = test_predictions['XGBoost']
y_mlp = test_predictions['MLP']
y_ens = test_predictions['Weighted Ensemble']

print(f'Predictions generated in {pred_elapsed:.2f} seconds ({len(test_df)/pred_elapsed:,.0f} rows/sec)')

baseline_metrics = evaluator.evaluate_predictions(test_predictions)
display_metrics = baseline_metrics[['Model', 'MAE', 'RMSE', 'R2', 'Samples']].copy()
print(display_metrics.to_string(index=False))

Predictions generated in 4.60 seconds (92,011 rows/sec)
            Model      MAE     RMSE       R2  Samples
    Random Forest 0.051427 0.201558 0.999870   422943
          XGBoost 0.161365 0.436400 0.999389   422943
Weighted Ensemble 0.309638 0.415459 0.999446   422943
Hybrid (Weighted) 0.309638 0.415459 0.999446   422943
              MLP 1.379964 1.671957 0.991029   422943


# Section 4 — MLP Integrated Gradients Analysis

In [4]:
def compute_integrated_gradients(model, x, baseline, steps=50):
    diff = x - baseline
    alphas = torch.linspace(0.0, 1.0, steps, device=x.device)
    total_grads = torch.zeros_like(x)
    for alpha in alphas:
        x_step = (baseline + alpha * diff).detach().requires_grad_(True)
        out = model(x_step)
        grad = torch.autograd.grad(out.sum(), x_step)[0]
        total_grads += grad
    return diff * (total_grads / steps)

sample_indices = np.random.RandomState(SEED).choice(len(test_df), size=IG_SAMPLE_SIZE, replace=False)
test_sample_df = test_df.iloc[sample_indices].copy().reset_index(drop=True)

_, _, x_sample_tensor, _ = evaluator.prepare_inputs(test_sample_df)
x_sample_tensor = x_sample_tensor.to(device)

baseline_tensor = torch.zeros(1, len(all_feature_names), device=device)

evaluator.mlp_model.eval()
t0 = time.time()
ig_attributions_tensor = compute_integrated_gradients(
    evaluator.mlp_model, x_sample_tensor, baseline_tensor, steps=IG_STEPS
)
ig_elapsed = time.time() - t0

ig_values = ig_attributions_tensor.detach().cpu().numpy()

with torch.no_grad():
    f_x = evaluator.mlp_model(x_sample_tensor).cpu().numpy().flatten()
    f_base = evaluator.mlp_model(baseline_tensor).cpu().numpy().flatten()

convergence_error = np.abs(ig_values.sum(axis=1) - (f_x - f_base)).mean()
print(f'Integrated Gradients computed for {IG_SAMPLE_SIZE} samples in {ig_elapsed:.2f}s')
print(f'Completeness mean absolute divergence: {convergence_error:.6f}')

mean_abs_attributions = np.abs(ig_values).mean(axis=0)
mean_signed_attributions = ig_values.mean(axis=0)
std_signed_attributions = ig_values.std(axis=0)

global_attribution_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Mean_Absolute_Attribution': mean_abs_attributions,
    'Mean_Signed_Attribution': mean_signed_attributions,
    'Std_Signed_Attribution': std_signed_attributions,
}).sort_values('Mean_Absolute_Attribution', ascending=False).reset_index(drop=True)

global_attribution_df.to_csv(output_directory / 'mlp_global_feature_attributions.csv', index=False)

country_ohe_cols = [c for c in one_hot_feature_names if c.startswith('country_code')]
wb_ohe_cols = [c for c in one_hot_feature_names if c.startswith('waterbody_type_code')]

aggregated_records = []
for num_f in numerical_features:
    idx = all_feature_names.index(num_f)
    aggregated_records.append({
        'Feature': num_f,
        'Group_Type': 'Numerical',
        'Mean_Absolute_Attribution': float(mean_abs_attributions[idx]),
        'Mean_Signed_Attribution': float(mean_signed_attributions[idx]),
    })

c_indices = [all_feature_names.index(c) for c in country_ohe_cols]
aggregated_records.append({
    'Feature': 'country_code',
    'Group_Type': 'Categorical_Aggregated',
    'Mean_Absolute_Attribution': float(np.abs(ig_values[:, c_indices]).sum(axis=1).mean()),
    'Mean_Signed_Attribution': float(ig_values[:, c_indices].sum(axis=1).mean()),
})

wb_indices = [all_feature_names.index(c) for c in wb_ohe_cols]
aggregated_records.append({
    'Feature': 'waterbody_type_code',
    'Group_Type': 'Categorical_Aggregated',
    'Mean_Absolute_Attribution': float(np.abs(ig_values[:, wb_indices]).sum(axis=1).mean()),
    'Mean_Signed_Attribution': float(ig_values[:, wb_indices].sum(axis=1).mean()),
})

aggregated_attribution_df = pd.DataFrame(aggregated_records).sort_values(
    'Mean_Absolute_Attribution', ascending=False
).reset_index(drop=True)

aggregated_attribution_df.to_csv(output_directory / 'mlp_aggregated_feature_attributions.csv', index=False)
print('Top 10 Global Features (53 Encoded Representation):')
print(global_attribution_df.head(10).to_string(index=False))
print('\nTop 10 Global Features (38 Aggregated Representation):')
print(aggregated_attribution_df.head(10).to_string(index=False))

Integrated Gradients computed for 1000 samples in 0.38s
Completeness mean absolute divergence: 0.239890
Top 10 Global Features (53 Encoded Representation):
             Feature  Mean_Absolute_Attribution  Mean_Signed_Attribution  Std_Signed_Attribution
log1p_orthophosphate                   5.860206                -1.342795                9.768473
      log1p_nitrogen                   3.416984                 0.675886                4.342524
      log1p_np_ratio                   3.039933                -0.270483                3.942786
       log1p_ammonia                   1.709629                -1.131784                4.168304
  exceedance_counter                   0.624713                -0.244932                1.088557
           log1p_bod                   0.577422                -0.526365                1.548250
        ammonia_mg_l                   0.360696                -0.257133                1.731813
       nitrogen_mg_l                   0.331076                -0.23

# Section 5 — Global Feature Importance Visualizations

In [5]:
fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
top_20_features = global_attribution_df.head(20).iloc[::-1]
ax.barh(top_20_features['Feature'], top_20_features['Mean_Absolute_Attribution'], color='royalblue')
ax.set_title('Top 20 Input Features by Mean Absolute Integrated Gradients Attribution', fontweight='bold')
ax.set_xlabel('Mean |Attribution| (CCME WQI units)')
plt.tight_layout()
plt.savefig(output_directory / '01_mlp_global_feature_importance.png', bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
top_15_names = global_attribution_df.head(15)['Feature'].tolist()[::-1]
top_15_indices = [all_feature_names.index(f) for f in top_15_names]
top_15_distributions = [ig_values[:, idx] for idx in top_15_indices]

box = ax.boxplot(top_15_distributions, vert=False, tick_labels=top_15_names, patch_artist=True, showfliers=False)
for patch in box['boxes']:
    patch.set_facecolor('lightblue')
ax.axvline(0, color='red', linestyle='--', linewidth=1.2, alpha=0.8)
ax.set_title('Directional Distribution of Signed Attributions (Top 15 Predictors)', fontweight='bold')
ax.set_xlabel('Attribution Value (Positive = Higher Predicted WQI, Negative = Lower WQI)')
plt.tight_layout()
plt.savefig(output_directory / '02_mlp_signed_feature_attributions.png', bbox_inches='tight')
plt.close(fig)

print('Saved Figures: 01_mlp_global_feature_importance.png and 02_mlp_signed_feature_attributions.png')

Saved Figures: 01_mlp_global_feature_importance.png and 02_mlp_signed_feature_attributions.png


<string>:15: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.


# Section 6 — Individual Prediction Explanations

In [6]:
sample_actuals = test_sample_df[target_column].values
sample_preds = f_x
sample_errors = sample_preds - sample_actuals
sample_abs_errors = np.abs(sample_errors)

case_indices = {
    'Excellent Quality (Low Error)': int(np.where((sample_actuals > 90) & (sample_abs_errors < 0.5))[0][0]),
    'Poor Quality (Low Error)': int(np.where((sample_actuals < 55) & (sample_abs_errors < 1.0))[0][0]),
    'Typical Case (Median Error)': int(np.argsort(np.abs(sample_abs_errors - np.median(sample_abs_errors)))[0]),
    'Notable Overprediction': int(np.argsort(sample_errors)[0]),
    'Notable Underprediction': int(np.argsort(sample_errors)[-1]),
}

case_records = []
fig, axes = plt.subplots(len(case_indices), 1, figsize=(11, 15), dpi=150)

for plot_idx, (case_name, row_idx) in enumerate(case_indices.items()):
    row_ig = ig_values[row_idx]
    actual_val = float(sample_actuals[row_idx])
    pred_val = float(sample_preds[row_idx])
    err_val = float(sample_errors[row_idx])
    
    sorted_attr_indices = np.argsort(row_ig)
    top_neg_indices = sorted_attr_indices[:3]
    top_pos_indices = sorted_attr_indices[-3:][::-1]
    
    top_features_indices = list(top_neg_indices) + list(top_pos_indices)
    top_features_indices = sorted(list(set(top_features_indices)), key=lambda idx: row_ig[idx])
    
    case_feature_names = [all_feature_names[i] for i in top_features_indices]
    case_feature_values = [row_ig[i] for i in top_features_indices]
    colors = ['firebrick' if v < 0 else 'forestgreen' for v in case_feature_values]
    
    ax = axes[plot_idx]
    ax.barh(case_feature_names, case_feature_values, color=colors)
    ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(f'{case_name} | Actual: {actual_val:.2f} | Pred: {pred_val:.2f} | Error: {err_val:+.2f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Attribution Contribution')
    
    case_records.append({
        'Case_Type': case_name,
        'Actual_WQI': actual_val,
        'Predicted_WQI': pred_val,
        'Prediction_Error': err_val,
        'Top_Positive_Feature': all_feature_names[top_pos_indices[0]],
        'Top_Positive_Attribution': float(row_ig[top_pos_indices[0]]),
        'Top_Negative_Feature': all_feature_names[top_neg_indices[0]],
        'Top_Negative_Attribution': float(row_ig[top_neg_indices[0]]),
    })

plt.tight_layout()
plt.savefig(output_directory / '03_individual_prediction_explanations.png', bbox_inches='tight')
plt.close(fig)

individual_cases_df = pd.DataFrame(case_records)
individual_cases_df.to_csv(output_directory / 'individual_prediction_explanations.csv', index=False)
print(individual_cases_df.to_string(index=False))

                    Case_Type  Actual_WQI  Predicted_WQI  Prediction_Error Top_Positive_Feature  Top_Positive_Attribution    Top_Negative_Feature  Top_Negative_Attribution
Excellent Quality (Low Error)  100.000000     100.124756          0.124756       log1p_nitrogen                  5.581943          log1p_np_ratio                 -1.710616
     Poor Quality (Low Error)   44.895610      44.761414         -0.134196 log1p_orthophosphate                  1.793973 log1p_unionized_ammonia                -22.595711
  Typical Case (Median Error)  100.000000      98.699661         -1.300339       log1p_nitrogen                  7.237075          log1p_np_ratio                 -1.864135
       Notable Overprediction   93.188363      87.066116         -6.122247 log1p_orthophosphate                  3.566235      exceedance_counter                 -2.243233
      Notable Underprediction   70.762460      80.275108          9.512648 log1p_orthophosphate                  3.241062               log1

# Section 7 — Hybrid Model Architecture and Component Analysis

In [7]:
residual_ckpt_path = REPO_ROOT / 'Phase_2_Deep_Learning' / 'models' / 'residual_mlp_checkpoint.pt'
hybrid_ckpt_path = REPO_ROOT / 'Phase_2_Deep_Learning' / 'models' / 'hybrid_ensemble_checkpoint.pt'

residual_ckpt_exists = residual_ckpt_path.exists()
hybrid_ckpt_exists = hybrid_ckpt_path.exists()

print(f'ResidualMLP Checkpoint Available:  {residual_ckpt_exists}')
print(f'Hybrid Ensemble Checkpoint Avail: {hybrid_ckpt_exists}')
print('Active Multi-Paradigm Ensemble:    WeightedEnsemble (0.4 RF + 0.4 XGBoost + 0.2 MLP)')

base_tree_preds = 0.5 * (y_rf + y_xgb)
mlp_adjustments = y_ens - base_tree_preds

tree_abs_errors = np.abs(y_true - base_tree_preds)
ens_abs_errors = np.abs(y_true - y_ens)
error_deltas = ens_abs_errors - tree_abs_errors

improved_mask = error_deltas < 0
worsened_mask = error_deltas > 0
neutral_mask = error_deltas == 0

pct_improved = float(np.mean(improved_mask) * 100)
pct_worsened = float(np.mean(worsened_mask) * 100)
pct_neutral = float(np.mean(neutral_mask) * 100)

print(f'Ensemble Reduces Base Error:   {pct_improved:.2f}% of test cases')
print(f'Ensemble Increases Base Error: {pct_worsened:.2f}% of test cases')
print(f'Ensemble Neutral:              {pct_neutral:.2f}% of test cases')
print(f'Mean Absolute Adjustment:      {np.abs(mlp_adjustments).mean():.4f} CCME units')

hybrid_comp_sample_df = pd.DataFrame({
    'Actual_WQI': y_true[:1000],
    'Base_Tree_Prediction': base_tree_preds[:1000],
    'MLP_Prediction': y_mlp[:1000],
    'Ensemble_Prediction': y_ens[:1000],
    'MLP_Adjustment': mlp_adjustments[:1000],
    'Tree_Absolute_Error': tree_abs_errors[:1000],
    'Ensemble_Absolute_Error': ens_abs_errors[:1000],
    'Error_Reduction': -error_deltas[:1000],
})
hybrid_comp_sample_df.to_csv(output_directory / 'hybrid_component_analysis.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
axes[0].hist(mlp_adjustments, bins=60, color='teal', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of MLP Adjustments to Base Tree Predictions', fontweight='bold')
axes[0].set_xlabel('Adjustment Value (Ensemble − Base Tree)')
axes[0].set_ylabel('Observation Count')

sample_plot_indices = np.random.RandomState(SEED).choice(len(y_true), size=10000, replace=False)
axes[1].scatter(base_tree_preds[sample_plot_indices], y_ens[sample_plot_indices], alpha=0.15, s=6, color='darkslateblue')
axes[1].plot([0, 100], [0, 100], 'r--', label='Identity Line')
axes[1].set_title('Base Tree vs Final Weighted Ensemble Predictions (10k Sample)', fontweight='bold')
axes[1].set_xlabel('Base Tree Prediction')
axes[1].set_ylabel('Weighted Ensemble Prediction')
axes[1].legend()
plt.tight_layout()
plt.savefig(output_directory / '04_hybrid_component_contributions.png', bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
ax.scatter(np.abs(mlp_adjustments)[sample_plot_indices], -error_deltas[sample_plot_indices], alpha=0.15, s=8, color='darkcyan')
ax.axhline(0, color='red', linestyle='--', label='Zero Error Change')
ax.set_title('MLP Adjustment Magnitude vs Base Error Reduction', fontweight='bold')
ax.set_xlabel('|MLP Adjustment| (CCME Units)')
ax.set_ylabel('Error Reduction: Error(Base) − Error(Ensemble)')
ax.legend()
plt.tight_layout()
plt.savefig(output_directory / '06_correction_vs_base_error.png', bbox_inches='tight')
plt.close(fig)

ResidualMLP Checkpoint Available:  False
Hybrid Ensemble Checkpoint Avail: False
Active Multi-Paradigm Ensemble:    WeightedEnsemble (0.4 RF + 0.4 XGBoost + 0.2 MLP)
Ensemble Reduces Base Error:   13.16% of test cases
Ensemble Increases Base Error: 86.84% of test cases
Ensemble Neutral:              0.00% of test cases
Mean Absolute Adjustment:      0.2733 CCME units


# Section 8 — Overall Prediction Error Distributions

In [8]:
models_dict = {
    'Random Forest': y_rf,
    'XGBoost': y_xgb,
    'MLP': y_mlp,
    'Weighted Ensemble': y_ens,
}

error_summary_rows = []
model_residuals = {}
model_abs_errors = {}

for name, y_pred in models_dict.items():
    res = y_true - y_pred
    ae = np.abs(res)
    model_residuals[name] = res
    model_abs_errors[name] = ae
    
    error_summary_rows.append({
        'Model': name,
        'Mean_Residual': float(np.mean(res)),
        'Residual_Std': float(np.std(res)),
        'Median_AE': float(np.median(ae)),
        'MAE': float(np.mean(ae)),
        'RMSE': float(np.sqrt(np.mean(res ** 2))),
        'P90_AE': float(np.percentile(ae, 90)),
        'P95_AE': float(np.percentile(ae, 95)),
        'P99_AE': float(np.percentile(ae, 99)),
        'Max_AE': float(np.max(ae)),
    })

error_summary_df = pd.DataFrame(error_summary_rows).sort_values('MAE').reset_index(drop=True)
error_summary_df.to_csv(output_directory / 'model_error_summary_comparison.csv', index=False)
print('COMPREHENSIVE ERROR DIAGNOSTIC SUMMARY:')
print(error_summary_df.to_string(index=False))

model_names = list(models_dict.keys())
win_matrix = pd.DataFrame(index=model_names, columns=model_names, dtype=float)

for m1 in model_names:
    for m2 in model_names:
        if m1 == m2:
            win_matrix.loc[m1, m2] = 50.0
        else:
            win_pct = float(np.mean(model_abs_errors[m1] < model_abs_errors[m2]) * 100)
            win_matrix.loc[m1, m2] = win_pct

win_matrix.to_csv(output_directory / 'model_comparison_win_rate_matrix.csv')
print('\nPAIRWISE WIN-RATE MATRIX (% Test Cases Where Row Model Has Lower Absolute Error Than Column Model):')
print(win_matrix.round(2).to_string())

COMPREHENSIVE ERROR DIAGNOSTIC SUMMARY:
            Model  Mean_Residual  Residual_Std  Median_AE      MAE     RMSE   P90_AE   P95_AE   P99_AE    Max_AE
    Random Forest      -0.000859      0.201556   0.004181 0.051427 0.201558 0.124954 0.205616 0.573800 12.612820
          XGBoost      -0.000166      0.436400   0.047981 0.161365 0.436400 0.379364 0.600689 1.729241  9.276878
Weighted Ensemble       0.177367      0.375696   0.265323 0.309638 0.415459 0.561375 0.738932 1.371895  8.828306
              MLP       0.888884      1.416095   1.289307 1.379964 1.671957 2.501479 3.089527 4.857225 31.132713

PAIRWISE WIN-RATE MATRIX (% Test Cases Where Row Model Has Lower Absolute Error Than Column Model):
                   Random Forest  XGBoost    MLP  Weighted Ensemble
Random Forest              50.00    87.51  97.67              90.82
XGBoost                    12.49    50.00  93.50              77.78
MLP                         2.33     6.50  50.00               3.09
Weighted Ensemble     

# Section 9 — Error Distribution Visualizations

In [9]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
colors = {'Random Forest': 'forestgreen', 'XGBoost': 'darkorange', 'MLP': 'royalblue', 'Weighted Ensemble': 'purple'}

for name in models_dict.keys():
    sns.kdeplot(model_residuals[name], ax=ax, label=name, color=colors[name], clip=(-10, 10), common_norm=False)

ax.set_title('Residual Density Comparison (y_true − y_pred)', fontweight='bold')
ax.set_xlabel('Residual (Positive = Underprediction, Negative = Overprediction)')
ax.set_ylabel('Density')
ax.set_xlim(-8, 8)
ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
ax.legend()
plt.tight_layout()
plt.savefig(output_directory / '07_residual_distributions_comparison.png', bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
box_data = [model_residuals[name] for name in models_dict.keys()]
box = ax.boxplot(box_data, tick_labels=list(models_dict.keys()), patch_artist=True, showfliers=False)
for patch, name in zip(box['boxes'], models_dict.keys()):
    patch.set_facecolor(colors[name])
    patch.set_alpha(0.7)
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('Comparative Residual Boxplots Across Tested Architectures', fontweight='bold')
ax.set_ylabel('Residual (y_true − y_pred)')
plt.tight_layout()
plt.savefig(output_directory / '08_residual_boxplots.png', bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(12, 11), dpi=150)
axes = axes.flatten()

for idx, (name, y_pred) in enumerate(models_dict.items()):
    ax = axes[idx]
    ax.hexbin(y_pred, y_true, gridsize=50, cmap='Blues', mincnt=1, bins='log')
    ax.plot([0, 100], [0, 100], 'r--', label='Identity (y=x)')
    ax.set_title(f'{name} — Actual vs Predicted', fontweight='bold')
    ax.set_xlabel('Predicted CCME WQI')
    ax.set_ylabel('Actual CCME WQI')
    ax.set_xlim(25, 105)
    ax.set_ylim(25, 105)
    if idx == 0:
        ax.legend()

plt.tight_layout()
plt.savefig(output_directory / '09_actual_vs_predicted.png', bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(12, 11), dpi=150)
axes = axes.flatten()

for idx, (name, y_pred) in enumerate(models_dict.items()):
    ax = axes[idx]
    ax.hexbin(y_pred, model_residuals[name], gridsize=50, cmap='Purples', mincnt=1, bins='log')
    ax.axhline(0, color='red', linestyle='--')
    ax.set_title(f'{name} — Residual vs Predicted', fontweight='bold')
    ax.set_xlabel('Predicted CCME WQI')
    ax.set_ylabel('Residual (y_true − y_pred)')
    ax.set_xlim(25, 105)
    ax.set_ylim(-15, 15)

plt.tight_layout()
plt.savefig(output_directory / '10_residual_vs_predicted.png', bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
sorted_errors = {name: np.sort(model_abs_errors[name]) for name in models_dict.keys()}
cdf_eval = np.linspace(0, 1, len(y_true))

for name in models_dict.keys():
    ax.plot(sorted_errors[name], cdf_eval, label=name, color=colors[name], linewidth=1.5)

ax.set_xlim(0, 5)
ax.set_title('Cumulative Distribution Function (CDF) of Absolute Errors', fontweight='bold')
ax.set_xlabel('Absolute Error Threshold (CCME WQI units)')
ax.set_ylabel('Fraction of Test Population Below Threshold')
ax.axvline(1.0, color='grey', linestyle=':', label='1.0 Unit Error Threshold')
ax.legend()
plt.tight_layout()
plt.savefig(output_directory / '11_absolute_error_cdf.png', bbox_inches='tight')
plt.close(fig)

print('Saved Figures: 07, 08, 09, 10, 11 successfully.')

Saved Figures: 07, 08, 09, 10, 11 successfully.


# Section 10 — Subgroup Error Pattern Diagnostics

In [10]:
wqi_bins = [0, 45, 65, 80, 95, 100]
wqi_labels = ['Poor [0-45]', 'Marginal (45-65]', 'Fair (65-80]', 'Good (80-95]', 'Excellent (95-100]']
wqi_tiers = pd.cut(test_df[target_column], bins=wqi_bins, labels=wqi_labels, include_lowest=True)

tier_records = []
for tier in wqi_labels:
    tier_mask = (wqi_tiers == tier).values
    count = int(np.sum(tier_mask))
    if count == 0:
        continue
    for name in models_dict.keys():
        sub_res = model_residuals[name][tier_mask]
        sub_ae = model_abs_errors[name][tier_mask]
        tier_records.append({
            'WQI_Tier': tier,
            'Model': name,
            'Sample_Count': count,
            'Mean_Residual': float(np.mean(sub_res)),
            'Median_AE': float(np.median(sub_ae)),
            'MAE': float(np.mean(sub_ae)),
            'RMSE': float(np.sqrt(np.mean(sub_res ** 2))),
        })

wqi_tier_df = pd.DataFrame(tier_records)
wqi_tier_df.to_csv(output_directory / 'error_diagnostics_by_wqi_tier.csv', index=False)

country_records = []
for country in test_df['country'].unique():
    c_mask = (test_df['country'] == country).values
    count = int(np.sum(c_mask))
    if count < 100:
        continue
    for name in models_dict.keys():
        sub_res = model_residuals[name][c_mask]
        sub_ae = model_abs_errors[name][c_mask]
        country_records.append({
            'Country': country,
            'Model': name,
            'Sample_Count': count,
            'Mean_Residual': float(np.mean(sub_res)),
            'Median_AE': float(np.median(sub_ae)),
            'MAE': float(np.mean(sub_ae)),
            'RMSE': float(np.sqrt(np.mean(sub_res ** 2))),
        })

country_err_df = pd.DataFrame(country_records)
country_err_df.to_csv(output_directory / 'error_diagnostics_by_country.csv', index=False)

wb_records = []
for wb in test_df['waterbody_type'].unique():
    wb_mask = (test_df['waterbody_type'] == wb).values
    count = int(np.sum(wb_mask))
    if count < 100:
        continue
    for name in models_dict.keys():
        sub_res = model_residuals[name][wb_mask]
        sub_ae = model_abs_errors[name][wb_mask]
        wb_records.append({
            'Waterbody_Type': wb,
            'Model': name,
            'Sample_Count': count,
            'Mean_Residual': float(np.mean(sub_res)),
            'Median_AE': float(np.median(sub_ae)),
            'MAE': float(np.mean(sub_ae)),
            'RMSE': float(np.sqrt(np.mean(sub_res ** 2))),
        })

wb_err_df = pd.DataFrame(wb_records)
wb_err_df.to_csv(output_directory / 'error_diagnostics_by_waterbody.csv', index=False)

test_years = test_df['year'].values
year_bins = [1950, 1980, 1990, 2000, 2010, 2025]
year_labels = ['<1980', '1980-1989', '1990-1999', '2000-2009', '2010-2023']
temporal_periods = pd.cut(test_years, bins=year_bins, labels=year_labels, include_lowest=True)

temporal_records = []
for period in year_labels:
    t_mask = (temporal_periods == period)
    count = int(np.sum(t_mask))
    if count < 100:
        continue
    for name in models_dict.keys():
        sub_res = model_residuals[name][t_mask]
        sub_ae = model_abs_errors[name][t_mask]
        temporal_records.append({
            'Period': period,
            'Model': name,
            'Sample_Count': count,
            'Mean_Residual': float(np.mean(sub_res)),
            'Median_AE': float(np.median(sub_ae)),
            'MAE': float(np.mean(sub_ae)),
            'RMSE': float(np.sqrt(np.mean(sub_res ** 2))),
        })

temporal_err_df = pd.DataFrame(temporal_records)
temporal_err_df.to_csv(output_directory / 'error_diagnostics_by_temporal.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=150)
sns.barplot(data=wqi_tier_df, x='WQI_Tier', y='MAE', hue='Model', ax=axes[0], palette='viridis')
axes[0].set_title('Mean Absolute Error Across Regulatory CCME WQI Tiers', fontweight='bold')
axes[0].set_ylabel('MAE (CCME Units)')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=wqi_tier_df, x='WQI_Tier', y='Mean_Residual', hue='Model', ax=axes[1], palette='viridis')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Mean Residual Bias Across WQI Tiers (Positive=Under, Negative=Over)', fontweight='bold')
axes[1].set_ylabel('Mean Residual')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(output_directory / '12_error_by_wqi_tier.png', bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=150)
sns.barplot(data=country_err_df, x='Country', y='MAE', hue='Model', ax=axes[0], palette='magma')
axes[0].set_title('MAE by Jurisdiction (Country)', fontweight='bold')
axes[0].set_ylabel('MAE')

sns.barplot(data=wb_err_df[wb_err_df['Model'] == 'MLP'].sort_values('MAE', ascending=False), x='Waterbody_Type', y='MAE', ax=axes[1], color='steelblue')
axes[1].set_title('MLP MAE Across Aquatic Ecosystem Classifications', fontweight='bold')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.savefig(output_directory / '13_error_by_country_waterbody.png', bbox_inches='tight')
plt.close(fig)

print('Saved Figures: 12 and 13 successfully.')

Saved Figures: 12 and 13 successfully.


# Section 11 — Difficult Case Analysis

In [11]:
meta_columns = ['country', 'waterbody_type', 'ph', 'do_mg_l', 'bod_mg_l', 'ammonia_mg_l', 'orthophosphate_mg_l', 'exceedance_counter']

top_err_indices = np.argsort(model_abs_errors['MLP'])[-20:][::-1]
large_errors_df = test_df.iloc[top_err_indices][meta_columns].copy()
large_errors_df['Actual_WQI'] = y_true[top_err_indices]
large_errors_df['MLP_Predicted'] = y_mlp[top_err_indices]
large_errors_df['MLP_Abs_Error'] = model_abs_errors['MLP'][top_err_indices]
large_errors_df['Ensemble_Predicted'] = y_ens[top_err_indices]
large_errors_df['Ensemble_Abs_Error'] = model_abs_errors['Weighted Ensemble'][top_err_indices]
large_errors_df['XGBoost_Predicted'] = y_xgb[top_err_indices]
large_errors_df.to_csv(output_directory / 'difficult_cases_largest_errors.csv', index=False)

top_over_indices = np.argsort(model_residuals['MLP'])[:20]
overpred_df = test_df.iloc[top_over_indices][meta_columns].copy()
overpred_df['Actual_WQI'] = y_true[top_over_indices]
overpred_df['MLP_Predicted'] = y_mlp[top_over_indices]
overpred_df['MLP_Residual'] = model_residuals['MLP'][top_over_indices]
overpred_df.to_csv(output_directory / 'difficult_cases_overpredictions.csv', index=False)

top_under_indices = np.argsort(model_residuals['MLP'])[-20:][::-1]
underpred_df = test_df.iloc[top_under_indices][meta_columns].copy()
underpred_df['Actual_WQI'] = y_true[top_under_indices]
underpred_df['MLP_Predicted'] = y_mlp[top_under_indices]
underpred_df['MLP_Residual'] = model_residuals['MLP'][top_under_indices]
underpred_df.to_csv(output_directory / 'difficult_cases_underpredictions.csv', index=False)

tradeoff_improve_indices = np.argsort(error_deltas)[:15]
tradeoff_worsen_indices = np.argsort(error_deltas)[-15:][::-1]

tradeoff_records = []
for idx in tradeoff_improve_indices:
    tradeoff_records.append({
        'Tradeoff_Type': 'Ensemble_Improvement',
        'Actual_WQI': float(y_true[idx]),
        'Base_Tree_Pred': float(base_tree_preds[idx]),
        'MLP_Pred': float(y_mlp[idx]),
        'Ensemble_Pred': float(y_ens[idx]),
        'Tree_Error': float(tree_abs_errors[idx]),
        'Ensemble_Error': float(ens_abs_errors[idx]),
        'Error_Improvement': float(-error_deltas[idx]),
        'Country': str(test_df.iloc[idx]['country']),
        'Waterbody': str(test_df.iloc[idx]['waterbody_type']),
    })

for idx in tradeoff_worsen_indices:
    tradeoff_records.append({
        'Tradeoff_Type': 'Ensemble_Degradation',
        'Actual_WQI': float(y_true[idx]),
        'Base_Tree_Pred': float(base_tree_preds[idx]),
        'MLP_Pred': float(y_mlp[idx]),
        'Ensemble_Pred': float(y_ens[idx]),
        'Tree_Error': float(tree_abs_errors[idx]),
        'Ensemble_Error': float(ens_abs_errors[idx]),
        'Error_Improvement': float(-error_deltas[idx]),
        'Country': str(test_df.iloc[idx]['country']),
        'Waterbody': str(test_df.iloc[idx]['waterbody_type']),
    })

tradeoff_df = pd.DataFrame(tradeoff_records)
tradeoff_df.to_csv(output_directory / 'difficult_cases_ensemble_tradeoffs.csv', index=False)

print('Representative Difficult Cases (Largest Absolute Errors):')
print(large_errors_df[['country', 'waterbody_type', 'Actual_WQI', 'MLP_Predicted', 'MLP_Abs_Error', 'Ensemble_Abs_Error']].head(5).to_string(index=False))

Representative Difficult Cases (Largest Absolute Errors):
country waterbody_type  Actual_WQI  MLP_Predicted  MLP_Abs_Error  Ensemble_Abs_Error
England       Effluent   67.715935      98.848648      31.132713            7.952940
England          River   57.570808      85.943573      28.372765            5.703251
England          River   50.203568      70.850983      20.647415            4.050804
    USA          River   52.417603      33.984066      18.433537            3.199826
England       Effluent   91.486977      74.093697      17.393280            3.202592


# Section 12 — Summary Artifacts and Final Verification

In [12]:
explainability_summary = {
    'integrated_gradients_sample_size': int(IG_SAMPLE_SIZE),
    'integrated_gradients_steps': int(IG_STEPS),
    'baseline_definition': 'zero_tensor_standardized_space_population_mean',
    'mean_completeness_divergence': float(convergence_error),
    'top_5_global_features_53dim': global_attribution_df.head(5)['Feature'].tolist(),
    'top_5_global_features_38dim': aggregated_attribution_df.head(5)['Feature'].tolist(),
    'top_feature_name': str(global_attribution_df.iloc[0]['Feature']),
    'top_feature_importance': float(global_attribution_df.iloc[0]['Mean_Absolute_Attribution']),
}

with open(output_directory / 'explainability_summary.json', 'w') as f:
    json.dump(explainability_summary, f, indent=2)

error_analysis_summary = {
    'total_test_observations': int(len(test_df)),
    'models_analyzed': list(models_dict.keys()),
    'model_mae': {name: float(np.mean(model_abs_errors[name])) for name in models_dict.keys()},
    'model_rmse': {name: float(np.sqrt(np.mean(model_residuals[name] ** 2))) for name in models_dict.keys()},
    'model_mean_residual': {name: float(np.mean(model_residuals[name])) for name in models_dict.keys()},
    'mlp_p95_error': float(np.percentile(model_abs_errors['MLP'], 95)),
    'mlp_max_error': float(np.max(model_abs_errors['MLP'])),
    'ensemble_improves_tree_pct': pct_improved,
    'residual_mlp_checkpoint_found': residual_ckpt_exists,
}

with open(output_directory / 'error_analysis_summary.json', 'w') as f:
    json.dump(error_analysis_summary, f, indent=2)

generated_files = sorted(os.listdir(output_directory))
print('=' * 65)
print('TASK 6 EXPLAINABILITY & PREDICTION ERROR ANALYSIS COMPLETE')
print('=' * 65)
print(f'Total Generated Artifacts: {len(generated_files)}')
for f in generated_files:
    size_kb = (output_directory / f).stat().st_size / 1024
    print(f'  ✓ {f:<42} ({size_kb:>7.1f} KB)')
print('=' * 65)

TASK 6 EXPLAINABILITY & PREDICTION ERROR ANALYSIS COMPLETE
Total Generated Artifacts: 28
  ✓ 01_mlp_global_feature_importance.png       (   68.2 KB)
  ✓ 02_mlp_signed_feature_attributions.png     (   74.0 KB)
  ✓ 03_individual_prediction_explanations.png  (  157.4 KB)
  ✓ 04_hybrid_component_contributions.png      (   91.7 KB)
  ✓ 06_correction_vs_base_error.png            (  112.7 KB)
  ✓ 07_residual_distributions_comparison.png   (   59.5 KB)
  ✓ 08_residual_boxplots.png                   (   28.8 KB)
  ✓ 09_actual_vs_predicted.png                 (  309.8 KB)
  ✓ 10_residual_vs_predicted.png               (  320.8 KB)
  ✓ 11_absolute_error_cdf.png                  (   83.7 KB)
  ✓ 12_error_by_wqi_tier.png                   (   97.6 KB)
  ✓ 13_error_by_country_waterbody.png          (   78.5 KB)
  ✓ difficult_cases_ensemble_tradeoffs.csv     (    4.9 KB)
  ✓ difficult_cases_largest_errors.csv         (    2.4 KB)
  ✓ difficult_cases_overpredictions.csv        (    1.5 KB)
  ✓ difficu